In [ ]:
# 1. Install Groq
!pip install -q groq streamlit pyngrok

# 2. Import libraries
from groq import Groq
import streamlit as st
from pyngrok import ngrok
import json
import time

# 3. Your API key (paste your real gsk_ key )
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
# 4. Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# 5. Test the API
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say OK in 2 words"}]
)
print("✅ Test response:", response.choices[0].message.content)

# 6. Define content brief function
def generate_content_brief(topic):
    prompt = f"""You are an expert SEO content strategist. For the topic: "{topic}"

Generate ONLY valid JSON with no extra text, following this exact structure:
{{
    "target_audience": ["persona 1", "persona 2", "persona 3"],
    "seo_keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
    "blog_outline": {{
        "h1_title": "Main title idea",
        "sections": ["Section 1 heading", "Section 2 heading", "Section 3 heading"]
    }},
    "tone_of_voice": "suggested tone"
}}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

# 7. Test the function
test_result = generate_content_brief("sustainable gardening")
print("\n📝 Test result:\n", test_result)

✅ Test response: It's okay

📝 Test result:
 {
    "target_audience": ["environmentally conscious homeowners", "urban gardeners", "eco-friendly landscapers"],
    "seo_keywords": ["sustainable gardening", "eco-friendly gardening practices", "organic gardening tips", "green gardening ideas", "environmental conservation"],
    "blog_outline": {
        "h1_title": "Cultivating a Greener Future: Sustainable Gardening Tips and Tricks",
        "sections": ["Introduction to Sustainable Gardening", "Eco-Friendly Gardening Practices", "Implementing Sustainable Gardening in Your Community"]
    },
    "tone_of_voice": "informative and encouraging"
}


In [ ]:
app_code_fixed = """
import streamlit as st
from groq import Groq
import json
import re

st.set_page_config(page_title="AI Content Brief Generator", layout="centered")
st.title("📝 AI Content Brief & SEO Assistant")
st.markdown("Generate a complete content brief for any topic using Llama 3 on Groq.")

# Paste your Groq API key here
GROQ_API_KEY = userdata.get('GROQ_API_KEY')  # <--- REPLACE with your actual gsk_ key

client = Groq(api_key=GROQ_API_KEY)

def extract_json(text):
    # Try to find JSON between ```json ... ``` or just raw {}
    match = re.search(r'```json\\s*(\\{.*?\\})\\s*```', text, re.DOTALL)
    if match:
        text = match.group(1)
    else:
        match = re.search(r'\\{.*\\}', text, re.DOTALL)
        if match:
            text = match.group(0)
    return text

topic = st.text_input("Enter your topic:", placeholder="e.g., sustainable gardening, remote work tools")

if st.button("Generate Content Brief"):
    if topic:
        with st.spinner("AI is working..."):
            prompt = f\"\"\"For topic: "{topic}"
Return ONLY valid JSON. No explanations. No markdown. Use this exact structure:
{{
    "target_audience": ["persona1", "persona2", "persona3"],
    "seo_keywords": ["kw1", "kw2", "kw3", "kw4", "kw5"],
    "blog_outline": {{
        "h1_title": "Main title",
        "sections": ["Section1", "Section2", "Section3"]
    }},
    "tone_of_voice": "professional"
}}
\"\"\"
            try:
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3
                )
                raw = response.choices[0].message.content
                cleaned = extract_json(raw)
                data = json.loads(cleaned)

                st.subheader("🎯 Target Audience")
                for p in data["target_audience"]:
                    st.write(f"- {p}")
                st.subheader("🔑 SEO Keywords")
                st.write(", ".join(data["seo_keywords"]))
                st.subheader("📚 Blog Outline")
                st.write(f"**H1:** {data['blog_outline']['h1_title']}")
                for i, sec in enumerate(data['blog_outline']['sections'], 1):
                    st.write(f"{i}. {sec}")
                st.subheader("🎭 Tone of Voice")
                st.write(data["tone_of_voice"])
            except Exception as e:
                st.error(f"Error: {e}")
                st.code(raw if 'raw' in locals() else "No response")
    else:
        st.warning("Please enter a topic.")
"""

# Write the file
with open("app.py", "w") as f:
    f.write(app_code_fixed)
print("✅ app.py updated with JSON error handling")

✅ app.py updated with JSON error handling


In [ ]:
import subprocess
import time
import os
from pyngrok import ngrok

os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(2)

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

print(ngrok.connect(8501))

NgrokTunnel: "https://proximity-armhole-decimal.ngrok-free.dev" -> "http://localhost:8501"
